# 06 — LIME Revisited

LIME was the first widely-adopted model-agnostic local explainer. Understanding its strengths and failure modes is essential for production use.

## Faithfulness vs Stability Tradeoff

LIME (Local Interpretable Model-agnostic Explanations, Ribeiro et al., 2016) fits a local linear surrogate around each prediction:

1. **Perturb** the input by sampling random neighbour points
2. **Query** the black-box model on those neighbours
3. **Weight** neighbours by proximity (using an exponential kernel)
4. **Fit** a sparse linear model (Lasso) on the weighted neighbours
5. **Return** the linear coefficients as feature importances

**Faithfulness problem**: the local linear approximation may be a poor fit if the decision boundary is highly non-linear in the neighbourhood. LIME's explanation is for the *surrogate*, not the model.

**Stability problem**: LIME's stochastic neighbourhood sampling means repeated runs can produce different explanations for the same input. Variance can be reduced by increasing `n_samples`, but at compute cost.

**Neighbourhood sampling**: the kernel width *σ* controls how large the local neighbourhood is. Too large → the surrogate captures global rather than local behaviour. Too small → variance explodes from sparse samples.

In [ ]:
# LIME implementation for tabular data
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=8, n_informative=4, random_state=42)
scaler = StandardScaler()
X = scaler.fit_transform(X)
feature_names = [f'f{i}' for i in range(8)]

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X[:800], y[:800])
print(f'RF accuracy: {clf.score(X[800:], y[800:]):.3f}')


class LIMETabular:
    def __init__(self, training_data, model_fn, kernel_width=0.75):
        self.training_data = training_data
        self.model_fn = model_fn
        self.kernel_width = kernel_width
        self.std = training_data.std(axis=0) + 1e-6

    def explain(self, x, n_samples=1000, n_features=None):
        n_feat = x.shape[0]
        if n_features is None:
            n_features = n_feat

        # Sample perturbed points
        noise = np.random.normal(0, 1, size=(n_samples, n_feat))
        Z = x + noise * self.std
        Z[0] = x  # include original

        # Query model
        preds = self.model_fn(Z)[:, 1]

        # Proximity weights
        distances = np.sqrt(((Z - x) ** 2 / self.std**2).sum(axis=1))
        weights = np.exp(-distances**2 / (2 * self.kernel_width**2))

        # Fit weighted linear model
        surrogate = Ridge(alpha=0.01)
        surrogate.fit(Z, preds, sample_weight=weights)

        # Return top n_features
        coefs = surrogate.coef_
        top_idx = np.argsort(np.abs(coefs))[::-1][:n_features]
        return {feature_names[i]: coefs[i] for i in top_idx}

lime = LIMETabular(X[:800], clf.predict_proba)
explanation = lime.explain(X[800], n_samples=2000)
print('\nLIME explanation for test sample 0:')
for feat, coef in sorted(explanation.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {feat}: {coef:+.4f}')

In [ ]:
# Stability analysis: how much does LIME vary across runs?
import matplotlib.pyplot as plt

n_runs = 20
all_coefs = []
for _ in range(n_runs):
    expl = lime.explain(X[800], n_samples=500)
    all_coefs.append([expl.get(f'f{i}', 0) for i in range(8)])

all_coefs = np.array(all_coefs)
means = all_coefs.mean(axis=0)
stds = all_coefs.std(axis=0)

fig, ax = plt.subplots(figsize=(9, 5))
x_pos = np.arange(8)
ax.bar(x_pos, means, yerr=stds, capsize=4, color='steelblue', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names)
ax.set_ylabel('LIME coefficient (mean ± std)')
ax.set_title(f'LIME Stability — {n_runs} runs, n_samples=500')
plt.tight_layout()
plt.savefig('/tmp/lime_stability.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nStability (CV = std/|mean|) per feature:')
for i in range(8):
    cv = stds[i] / (abs(means[i]) + 1e-6)
    print(f'  f{i}: mean={means[i]:+.3f}, std={stds[i]:.3f}, CV={cv:.2f}')

In [ ]:
# LIME on text (bag-of-words)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Toy text dataset
texts = [
    'the stock market is rising', 'profit up earnings beat',
    'company reports losses', 'market crash stocks fall',
    'investors cheer quarterly results', 'bankruptcy filed company closed',
    'revenue growth strong guidance', 'debt default warning issued',
]
labels = [1, 1, 0, 0, 1, 0, 1, 0]  # 1=positive, 0=negative

vectorizer = TfidfVectorizer()
X_text = vectorizer.fit_transform(texts).toarray()
lr_text = LogisticRegression(max_iter=1000)
lr_text.fit(X_text, labels)

def lime_text_explain(text, model, vectorizer, n_samples=500):
    words = text.split()
    n_words = len(words)
    if n_words == 0:
        return {}

    # Sample word masks
    masks = np.random.randint(0, 2, size=(n_samples, n_words))
    masks[0] = 1

    # Build masked texts and get predictions
    masked_texts = [' '.join([w for w, m in zip(words, mask) if m]) or 'neutral'
                    for mask in masks]
    X_masked = vectorizer.transform(masked_texts).toarray()
    preds = model.predict_proba(X_masked)[:, 1]

    # Proximity weights
    n_present = masks.sum(axis=1)
    weights = np.exp(-(n_words - n_present)**2 / (2 * (n_words * 0.25)**2))

    # Weighted ridge
    surrogate = Ridge(alpha=0.1)
    surrogate.fit(masks, preds, sample_weight=weights)
    return dict(zip(words, surrogate.coef_))

test_text = 'profit growth strong but debt warning'
explanation = lime_text_explain(test_text, lr_text, vectorizer)
print(f'LIME text explanation for: "{test_text}"')
for word, coef in sorted(explanation.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f'  "{word}": {coef:+.4f}')